# 04 — Mint-source minus clean-background raster

This notebook compares the existing clean-background raster `line_raster_blank_fresh_01` with the later blotter retry `line_raster_mint_blotter_retry_01`. Both used the same sequential monotonic serpentine pattern in the same desk coordinate frame. The mint treatment was a horizontal paper strip on a ziplock barrier; post-run operator notes report probable oversaturation and an unknown exact dose, so the result is a background-referenced geometry pilot rather than a reproducible concentration experiment.

The comparison keeps the independently estimated **3.0 s** physical response lag fixed. Each run uses its own late-flag-1 reference. Scan boundaries come from trajectory/video review rather than odor magnitude. Direct subtraction is evaluated only on grid cells supported by both trajectories. The physical source was marked and photographed, but its desk-coordinate center and endpoints were not entered, so this notebook does not claim a localization error or recovered line length.

In [1]:
from pathlib import Path
import bisect
import csv
import json
import math
import statistics
import sys

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'experiments').is_dir() and (candidate / 'record_cyranose_reading_pose.py').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the realsense-apriltag repository.')

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.response_pipeline import TrialConfig, find_session_by_trial_id

RESPONSE_LAG_S = 3.0
HEIGHT_BAND_CM = (1.0, 3.5)
MAX_INTERPOLATION_GAP_S = 1.5
DESK_BOUNDS = (13.0, 38.0, 18.0, 47.0)

# Pose-clock intervals determined from trajectory/video start and stop, not sensor response.
TRIALS = {
    'Blank': {'trial_id': 'line_raster_blank_fresh_01', 'pose_window_s': (86.0, 198.0)},
    'Mint': {'trial_id': 'line_raster_mint_blotter_retry_01', 'pose_window_s': (24.0, 127.0)},
}

for name, config in TRIALS.items():
    config['path'] = find_session_by_trial_id(ROOT, config['trial_id'])
    config['config'] = TrialConfig(
        trial_id=config['trial_id'],
        lag_s=RESPONSE_LAG_S,
        height_band_cm=HEIGHT_BAND_CM,
        desk_bounds_cm=DESK_BOUNDS,
        pose_window_s=config['pose_window_s'],
        max_interpolation_gap_s=MAX_INTERPOLATION_GAP_S,
    )
    print(config['trial_id'], '->', config['path'].relative_to(ROOT))


line_raster_blank_fresh_01 -> experiments/spatial-mapping/line-raster/matched-pairs/clean-block-01/cyranose_reading_pose_session_20260722_161100
line_raster_mint_blotter_retry_01 -> experiments/spatial-mapping/line-raster/background-referenced/blotter-retry-01


## Normalize and align

For sensor $i$, the late flag-1 median is $b_i$. Each reading becomes $r_i=100(s_i-b_i)/b_i$, and RMS response is $\sqrt{\frac{1}{32}\sum_i r_i^2}$. The reading at time $t$ is assigned to the interpolated pose at $t-3.0$ seconds.

In [2]:
from analysis.response_pipeline import (
    load_session, qualifying_rows, median_fingerprint, summarize_responses, percentile,
)

def load_trial(config):
    session = load_session(config['path'])
    rows = qualifying_rows(session.raw_rows, config['config'])
    row_dicts = [
        {'x': row.x, 'y': row.y, 'z': row.z, 'height': row.height_cm,
         'response': row.response, 'fractional': row.fractional}
        for row in rows
    ]
    summary = summarize_responses([row.response for row in rows])
    return {
        'metadata': session.metadata,
        'alignment': session.alignment,
        'rows': row_dicts,
        'fingerprint': median_fingerprint(
            [row.fractional for row in rows], len(config['config'].sensor_fields)
        ),
        'response_median': summary['response_median'],
        'response_p90': summary['response_p90'],
        'height_median': statistics.median(row.height_cm for row in rows),
    }

results = {name: load_trial(config) for name, config in TRIALS.items()}
for name, result in results.items():
    alignment = result['alignment']
    print(
        f"{name}: {alignment['matched_readings']}/{alignment['total_readings']} digitally matched; "
        f"p95 {alignment['absolute_pose_minus_pcnose_ms']['p95']:.1f} ms; "
        f"{len(result['rows'])} retained spatial readings; median height {result['height_median']:.2f} cm"
    )
    print(f"  RMS median {result['response_median']:.3f}% | p90 {result['response_p90']:.3f}%")

blank_vector = results['Blank']['fingerprint']
mint_vector = results['Mint']['fingerprint']
dot = sum(a * b for a, b in zip(blank_vector, mint_vector))
blank_norm = math.sqrt(sum(value * value for value in blank_vector))
mint_norm = math.sqrt(sum(value * value for value in mint_vector))
fingerprint_cosine = dot / (blank_norm * mint_norm)
overall_rms_ratio = results['Mint']['response_median'] / results['Blank']['response_median']
print(f'Mint/blank median RMS ratio: {overall_rms_ratio:.2f}x')
print(f'Median signed fingerprint cosine similarity: {fingerprint_cosine:.3f}')


Blank: 394/394 digitally matched; p95 93.6 ms; 174 retained spatial readings; median height 1.84 cm
  RMS median 0.071% | p90 0.083%
Mint: 268/268 digitally matched; p95 93.1 ms; 126 retained spatial readings; median height 2.56 cm
  RMS median 0.282% | p90 0.722%
Mint/blank median RMS ratio: 3.99x
Median signed fingerprint cosine similarity: -0.930


## Shared-grid maps and direct subtraction

Both trajectories are Gaussian-smoothed onto the same 1 cm desk grid using a 3 cm neighborhood. Blank, mint, and mint-minus-blank are displayed only where both scans contribute at least two nearby readings. Blank and mint share one absolute RMS color scale.

In [3]:
from analysis.response_pipeline import smooth_grid as _smooth_grid

GRID_X = [float(value) for value in range(13, 39)]
GRID_Y = [float(value) for value in range(18, 48)]
SMOOTH_RADIUS_CM = 3.0
SMOOTH_SIGMA_CM = 1.7

def smooth_grid(points):
    return _smooth_grid(
        points, GRID_X, GRID_Y,
        radius_cm=SMOOTH_RADIUS_CM, sigma_cm=SMOOTH_SIGMA_CM, min_count=2,
        x_of=lambda p: p['x'], y_of=lambda p: p['y'], value_of=lambda p: p['response'],
    )

blank_grid = smooth_grid(results['Blank']['rows'])
mint_grid = smooth_grid(results['Mint']['rows'])
blank_common, mint_common, difference_grid = [], [], []
difference_values = []
for blank_row, mint_row in zip(blank_grid, mint_grid):
    blank_out, mint_out, difference_out = [], [], []
    for blank_value, mint_value in zip(blank_row, mint_row):
        if blank_value is None or mint_value is None:
            blank_out.append(None); mint_out.append(None); difference_out.append(None)
        else:
            blank_out.append(blank_value); mint_out.append(mint_value)
            difference = mint_value - blank_value
            difference_out.append(difference); difference_values.append(difference)
    blank_common.append(blank_out); mint_common.append(mint_out); difference_grid.append(difference_out)

shared_values = [value for grid in (blank_common, mint_common) for row in grid for value in row if value is not None]
shared_max = percentile(shared_values, 0.98)
difference_limit = percentile([abs(value) for value in difference_values], 0.98)
positive_fraction = sum(value > 0 for value in difference_values) / len(difference_values)
median_difference = statistics.median(difference_values)

def profile_by_y(grid):
    profile = []
    for gy, row in zip(GRID_Y, grid):
        values = [value for value in row if value is not None]
        profile.append((gy, None if len(values) < 5 else statistics.median(values)))
    return profile

def profile_by_x(grid):
    profile = []
    for index, gx in enumerate(GRID_X):
        values = [row[index] for row in grid if row[index] is not None]
        profile.append((gx, None if len(values) < 5 else statistics.median(values)))
    return profile

difference_y_profile = profile_by_y(difference_grid)
difference_x_profile = profile_by_x(difference_grid)
valid_y_profile = [(position, value) for position, value in difference_y_profile if value is not None]
detected_band_y, detected_band_peak = max(valid_y_profile, key=lambda item: item[1])
half_peak_positions = [position for position, value in valid_y_profile if value >= detected_band_peak / 2]
detected_halfmax_width = max(half_peak_positions) - min(half_peak_positions) if half_peak_positions else float('nan')

fig = make_subplots(rows=1, cols=3, subplot_titles=('Clean blank', 'Mint', 'Mint minus blank'))
fig.add_trace(go.Heatmap(x=GRID_X, y=GRID_Y, z=blank_common, coloraxis='coloraxis', hovertemplate='X %{x:.1f} cm<br>Y %{y:.1f} cm<br>RMS %{z:.3f}%<extra></extra>'), row=1, col=1)
fig.add_trace(go.Heatmap(x=GRID_X, y=GRID_Y, z=mint_common, coloraxis='coloraxis', hovertemplate='X %{x:.1f} cm<br>Y %{y:.1f} cm<br>RMS %{z:.3f}%<extra></extra>'), row=1, col=2)
fig.add_trace(go.Heatmap(x=GRID_X, y=GRID_Y, z=difference_grid, coloraxis='coloraxis2', hovertemplate='X %{x:.1f} cm<br>Y %{y:.1f} cm<br>Difference %{z:+.3f} percentage points<extra></extra>'), row=1, col=3)
for column in (1, 2, 3):
    fig.update_xaxes(title_text='Desk X (cm)', autorange='reversed', row=1, col=column)
fig.update_yaxes(title_text='Desk Y (cm)', autorange='reversed', scaleanchor='x', scaleratio=1, row=1, col=1)
fig.update_yaxes(autorange='reversed', scaleanchor='x2', scaleratio=1, row=1, col=2)
fig.update_yaxes(autorange='reversed', scaleanchor='x3', scaleratio=1, row=1, col=3)
fig.add_hline(y=detected_band_y, line_dash='dash', line_color='#344563', annotation_text='difference-profile peak', row=1, col=3)
fig.update_layout(
    title='Mint-source versus clean background | fixed 3.0 s lag | common spatial support only',
    width=1450, height=620, template='plotly_white', showlegend=False,
    coloraxis=dict(colorscale='Cividis', cmin=0, cmax=shared_max, colorbar=dict(title='RMS change (%)', x=0.65)),
    coloraxis2=dict(colorscale='RdBu_r', cmin=-difference_limit, cmax=difference_limit, colorbar=dict(title='Mint − blank<br>(percentage points)', x=1.01)),
    margin=dict(l=60, r=170, t=90, b=60),
)
fig.show()
print(f'Common supported grid cells: {len(difference_values)}')
print(f'Median mint-minus-blank difference: {median_difference:.3f} percentage points')
print(f'Grid cells with mint > blank: {100 * positive_fraction:.1f}%')
print(f'Difference-profile peak: Desk Y {detected_band_y:.1f} cm | approximate half-maximum width {detected_halfmax_width:.1f} cm')


Common supported grid cells: 712
Median mint-minus-blank difference: 0.215 percentage points
Grid cells with mint > blank: 96.3%
Difference-profile peak: Desk Y 34.0 cm | approximate half-maximum width 14.0 cm


## Difference profiles and sensor direction

The first two panels summarize the direct subtraction across each desk axis. Both rasters progressed from larger toward smaller Desk X, so a strong X-gradient may include temporal accumulation or recovery as well as spatial structure. The third panel retains the sign of each sensor's median percentage change.

In [4]:
fig2 = make_subplots(rows=1, cols=3, subplot_titles=('Mint − blank by Desk Y', 'Mint − blank by Desk X', 'Median signed 32-sensor fingerprint'))
fig2.add_trace(go.Scatter(x=[p for p, v in difference_y_profile if v is not None], y=[v for p, v in difference_y_profile if v is not None], mode='lines+markers', line=dict(color='#5B6DEE', width=3), marker=dict(size=7), showlegend=False), row=1, col=1)
fig2.add_trace(go.Scatter(x=[p for p, v in difference_x_profile if v is not None], y=[v for p, v in difference_x_profile if v is not None], mode='lines+markers', line=dict(color='#5B6DEE', width=3), marker=dict(size=7), showlegend=False), row=1, col=2)
styles = {'Blank': ('#6B7280', 'diamond'), 'Mint': ('#5B6DEE', 'circle')}
for name, result in results.items():
    color, symbol = styles[name]
    fig2.add_trace(go.Scatter(x=list(range(1, 33)), y=result['fingerprint'], mode='lines+markers', name=name, line=dict(color=color, width=2), marker=dict(symbol=symbol, size=6)), row=1, col=3)
fig2.add_hline(y=0, line_dash='dot', line_color='#687386', row=1, col=1)
fig2.add_hline(y=0, line_dash='dot', line_color='#687386', row=1, col=2)
fig2.add_hline(y=0, line_dash='dot', line_color='#687386', row=1, col=3)
fig2.update_xaxes(title_text='Desk Y (cm)', row=1, col=1)
fig2.update_xaxes(title_text='Desk X (cm)', row=1, col=2)
fig2.update_xaxes(title_text='Cyranose sensor', dtick=4, row=1, col=3)
fig2.update_yaxes(title_text='Median RMS difference (percentage points)', row=1, col=1)
fig2.update_yaxes(title_text='Median RMS difference (percentage points)', row=1, col=2)
fig2.update_yaxes(title_text='Median signed change from reference (%)', row=1, col=3)
fig2.update_layout(width=1450, height=500, template='plotly_white', legend=dict(orientation='h', y=1.13, x=0.68), margin=dict(l=80, r=40, t=100, b=70))
fig2.show()

display(Markdown(
    f"**Background-referenced result.** Median RMS response was **{overall_rms_ratio:.2f}×** higher during mint-source than clean background. "
    f"Across the shared map, **{100 * positive_fraction:.1f}%** of supported cells had mint above blank, with a median direct difference of "
    f"**{median_difference:.3f} percentage points**. The difference profile formed a band peaking at **Desk Y {detected_band_y:.1f} cm** "
    f"with an approximate **{detected_halfmax_width:.1f} cm** half-maximum width after smoothing. "
    f"The signed 32-sensor fingerprints pointed in strongly different directions (cosine **{fingerprint_cosine:.3f}**)."
    + chr(10) * 2
    + "This retry is a useful geometry pilot, but the source center/endpoints were not entered in desk coordinates, the exact oil dose is unknown and reportedly excessive, and Desk X remains partly confounded with scan time. Treat the subtraction as mint-source versus clean background, not final line-reconstruction or concentration validation."
))

**Background-referenced result.** Median RMS response was **3.99×** higher during mint-source than clean background. Across the shared map, **96.3%** of supported cells had mint above blank, with a median direct difference of **0.215 percentage points**. The difference profile formed a band peaking at **Desk Y 34.0 cm** with an approximate **14.0 cm** half-maximum width after smoothing. The signed 32-sensor fingerprints pointed in strongly different directions (cosine **-0.930**).

This retry is a useful geometry pilot, but the source center/endpoints were not entered in desk coordinates, the exact oil dose is unknown and reportedly excessive, and Desk X remains partly confounded with scan time. Treat the subtraction as mint-source versus clean background, not final line-reconstruction or concentration validation.

## Diagonal-shape diagnosis

The background-referenced pair separates mint-source from clean background, but source loading and height consistency must be audited before interpreting geometry. The cells below test whether the hotspot appearance is associated with height filtering, coverage, scan order, smoothing, or the assumed physical lag. These are sensitivity checks, not parameter tuning: every planned setting is shown. Top-down spatial plots are rotated 180 degrees so the operator's physical start appears at the top-left; Desk X and Desk Y values remain unchanged.

In [5]:
import dataclasses

from analysis.response_pipeline import load_session, qualifying_rows

def _to_diagnostic_dict(row):
    return {
        'x': row.x, 'y': row.y, 'z': row.z,
        'height': row.height_cm, 'height_in_band': row.height_in_band,
        'response': row.response, 'fractional': row.fractional,
        'reading_time_s': row.reading_time_s, 'pose_time_s': row.pose_time_s,
        'scan_elapsed_s': row.scan_elapsed_s, 'scan_fraction': row.scan_fraction,
    }

def corrected_diagnostic_rows(source, lag_s, apply_height_filter=True):
    config = dataclasses.replace(source['config'], lag_s=lag_s)
    rows = qualifying_rows(source['raw_rows'], config, apply_height_filter=apply_height_filter)
    return [_to_diagnostic_dict(row) for row in rows]

diagnostic_sources = {
    name: {'raw_rows': load_session(config['path']).raw_rows, 'config': config['config']}
    for name, config in TRIALS.items()
}
diagnostic_rows_all_heights = {name: corrected_diagnostic_rows(source, RESPONSE_LAG_S, apply_height_filter=False) for name, source in diagnostic_sources.items()}
diagnostic_rows = {name: [row for row in diagnostic_rows_all_heights[name] if row['height_in_band']] for name in diagnostic_rows_all_heights}
for name in ('Blank', 'Mint'):
    excluded = [row for row in diagnostic_rows_all_heights[name] if not row['height_in_band']]
    print(f'{name}: {len(diagnostic_rows[name])} retained; {len(excluded)} excluded by the 1-3.5 cm height band; original analysis retained {len(results[name]["rows"])}')


Blank: 174 retained; 0 excluded by the 1-3.5 cm height band; original analysis retained 174
Mint: 126 retained; 21 excluded by the 1-3.5 cm height band; original analysis retained 126


### 1. Raw lag-corrected observations

Each colored marker below is one retained Cyranose reading at its lag-corrected pose. No spatial grid or smoothing is used. Red open X markers identify readings removed only because the reconstructed snout height was outside 1-3.5 cm. The thin line uses all in-bounds poses, including excluded heights, so gaps in colored sampling are visible rather than silently connected.

In [6]:
raw_responses = [row['response'] for rows in diagnostic_rows_all_heights.values() for row in rows]
raw_color_max = percentile(raw_responses, 0.98)
fig_raw = make_subplots(rows=1, cols=2, subplot_titles=('Blank: raw readings', 'Mint: raw readings'))
for column, name in enumerate(('Blank', 'Mint'), start=1):
    rows = diagnostic_rows[name]
    all_rows = diagnostic_rows_all_heights[name]
    excluded_rows = [row for row in all_rows if not row['height_in_band']]
    fig_raw.add_trace(go.Scatter(
        x=[row['x'] for row in all_rows], y=[row['y'] for row in all_rows], mode='lines',
        line=dict(color='#9AA4B2', width=1), hoverinfo='skip', showlegend=False,
    ), row=1, col=column)
    fig_raw.add_trace(go.Scatter(
        x=[row['x'] for row in rows], y=[row['y'] for row in rows], mode='markers',
        marker=dict(size=8, color=[row['response'] for row in rows], coloraxis='coloraxis', line=dict(width=0.4, color='#F3F4F6')),
        customdata=[[row['scan_elapsed_s'], row['height']] for row in rows],
        hovertemplate='X %{x:.1f} cm<br>Y %{y:.1f} cm<br>RMS %{marker.color:.3f}%<br>Scan time %{customdata[0]:.1f} s<br>Height %{customdata[1]:.2f} cm<extra></extra>',
        showlegend=False,
    ), row=1, col=column)
    if excluded_rows:
        fig_raw.add_trace(go.Scatter(
            x=[row['x'] for row in excluded_rows], y=[row['y'] for row in excluded_rows], mode='markers',
            marker=dict(size=11, symbol='x-open', color='#B42318', line=dict(width=2)),
            customdata=[[row['height'], row['scan_elapsed_s']] for row in excluded_rows],
            hovertemplate='Excluded by height<br>X %{x:.1f} cm<br>Y %{y:.1f} cm<br>Height %{customdata[0]:.2f} cm<br>Scan time %{customdata[1]:.1f} s<extra></extra>',
            name='Excluded: outside 1-3.5 cm', showlegend=(column == 2),
        ), row=1, col=column)
    fig_raw.add_trace(go.Scatter(
        x=[all_rows[0]['x'], all_rows[-1]['x']], y=[all_rows[0]['y'], all_rows[-1]['y']],
        mode='markers+text', text=['Start', 'End'], textposition=['top center', 'bottom center'],
        marker=dict(size=11, symbol=['triangle-up', 'square'], color='#26364A'),
        hoverinfo='skip', showlegend=False,
    ), row=1, col=column)
    fig_raw.update_xaxes(title_text='Desk X (cm)', autorange='reversed', row=1, col=column)
fig_raw.update_yaxes(title_text='Desk Y (cm)', autorange='reversed', scaleanchor='x', scaleratio=1, row=1, col=1)
fig_raw.update_yaxes(autorange='reversed', scaleanchor='x2', scaleratio=1, row=1, col=2)
fig_raw.update_layout(
    title='Raw measurements before gridding | fixed 3.0 s lag', width=1180, height=570, template='plotly_white',
    coloraxis=dict(colorscale='Cividis', cmin=0, cmax=raw_color_max, colorbar=dict(title='RMS change (%)')),
    margin=dict(l=70, r=130, t=90, b=60),
)
fig_raw.show()
for name in ('Blank', 'Mint'):
    all_rows = diagnostic_rows_all_heights[name]
    excluded_rows = [row for row in all_rows if not row['height_in_band']]
    if excluded_rows:
        print(f"{name}: {len(excluded_rows)}/{len(all_rows)} readings ({100 * len(excluded_rows) / len(all_rows):.1f}%) excluded by height; all excluded heights {min(row['height'] for row in excluded_rows):.2f}-{max(row['height'] for row in excluded_rows):.2f} cm at Desk Y {min(row['y'] for row in excluded_rows):.1f}-{max(row['y'] for row in excluded_rows):.1f} cm")
    else:
        print(f'{name}: 0/{len(all_rows)} readings excluded by height')

Blank: 0/174 readings excluded by height
Mint: 21/147 readings (14.3%) excluded by height; all excluded heights 3.50-7.46 cm at Desk Y 31.2-43.8 cm


### 1b. Snout-height audit

These maps include every lag-corrected reading inside the scan window and desk bounds before applying the height band. Crosses mark the points omitted from all response maps. A localized cluster of crosses means the measurement protocol and the analysis filter removed data systematically at that location.

In [7]:
all_heights = [row['height'] for rows in diagnostic_rows_all_heights.values() for row in rows]
height_color_max = max(HEIGHT_BAND_CM[1], percentile(all_heights, 0.98))
fig_height = make_subplots(rows=1, cols=2, subplot_titles=('Blank snout height', 'Mint snout height'))
for column, name in enumerate(('Blank', 'Mint'), start=1):
    rows = diagnostic_rows_all_heights[name]
    excluded_rows = [row for row in rows if not row['height_in_band']]
    fig_height.add_trace(go.Scatter(
        x=[row['x'] for row in rows], y=[row['y'] for row in rows], mode='lines+markers',
        line=dict(color='#A4ACB8', width=1),
        marker=dict(size=8, color=[row['height'] for row in rows], coloraxis='coloraxis', line=dict(width=0.4, color='#F3F4F6')),
        customdata=[[row['height'], row['height_in_band']] for row in rows],
        hovertemplate='X %{x:.1f} cm<br>Y %{y:.1f} cm<br>Height %{customdata[0]:.2f} cm<br>Retained %{customdata[1]}<extra></extra>',
        showlegend=False,
    ), row=1, col=column)
    if excluded_rows:
        fig_height.add_trace(go.Scatter(
            x=[row['x'] for row in excluded_rows], y=[row['y'] for row in excluded_rows], mode='markers',
            marker=dict(size=12, symbol='x-open', color='#B42318', line=dict(width=2)),
            name='Excluded by height', showlegend=(column == 2), hoverinfo='skip',
        ), row=1, col=column)
    fig_height.update_xaxes(title_text='Desk X (cm)', autorange='reversed', row=1, col=column)
fig_height.update_yaxes(title_text='Desk Y (cm)', autorange='reversed', scaleanchor='x', scaleratio=1, row=1, col=1)
fig_height.update_yaxes(autorange='reversed', scaleanchor='x2', scaleratio=1, row=1, col=2)
fig_height.update_layout(
    title='Snout height before applying the 1-3.5 cm filter', width=1180, height=570, template='plotly_white',
    coloraxis=dict(colorscale='Viridis', cmin=HEIGHT_BAND_CM[0], cmax=height_color_max, colorbar=dict(title='Height (cm)')),
    legend=dict(orientation='h', y=1.12, x=0.72), margin=dict(l=70, r=120, t=95, b=60),
)
fig_height.show()

### 2. Sampling density and common support

Counts use the same 3 cm neighborhood as the main heatmap. A comparison cell exists only when both runs contribute at least two nearby readings. White cells in the common-support panel are missing comparison coverage, not zero odor.

In [8]:
def count_grid(points, radius_cm):
    return [[
        sum((point['x'] - gx) ** 2 + (point['y'] - gy) ** 2 <= radius_cm ** 2 for point in points)
        for gx in GRID_X
    ] for gy in GRID_Y]

blank_counts = count_grid(diagnostic_rows['Blank'], SMOOTH_RADIUS_CM)
mint_counts = count_grid(diagnostic_rows['Mint'], SMOOTH_RADIUS_CM)
common_counts = []
coverage_differences = []
for blank_row, mint_row in zip(blank_counts, mint_counts):
    common_row = []
    for blank_count, mint_count in zip(blank_row, mint_row):
        common_row.append(min(blank_count, mint_count) if min(blank_count, mint_count) >= 2 else None)
        if min(blank_count, mint_count) >= 2:
            coverage_differences.append(abs(blank_count - mint_count))
    common_counts.append(common_row)
count_scale_max = percentile([value for grid in (blank_counts, mint_counts) for row in grid for value in row], 0.98)
fig_coverage = make_subplots(rows=1, cols=3, subplot_titles=('Blank nearby readings', 'Mint nearby readings', 'Common support: smaller count'))
for column, grid in enumerate((blank_counts, mint_counts, common_counts), start=1):
    fig_coverage.add_trace(go.Heatmap(
        x=GRID_X, y=GRID_Y, z=grid, coloraxis='coloraxis',
        hovertemplate='X %{x:.1f} cm<br>Y %{y:.1f} cm<br>Nearby readings %{z}<extra></extra>',
    ), row=1, col=column)
    fig_coverage.update_xaxes(title_text='Desk X (cm)', autorange='reversed', row=1, col=column)
fig_coverage.update_yaxes(title_text='Desk Y (cm)', autorange='reversed', scaleanchor='x', scaleratio=1, row=1, col=1)
fig_coverage.update_yaxes(autorange='reversed', scaleanchor='x2', scaleratio=1, row=1, col=2)
fig_coverage.update_yaxes(autorange='reversed', scaleanchor='x3', scaleratio=1, row=1, col=3)
fig_coverage.update_layout(
    title='Sampling support for the matched comparison', width=1450, height=600, template='plotly_white',
    coloraxis=dict(colorscale='Blues', cmin=0, cmax=count_scale_max, colorbar=dict(title='Readings within 3 cm')),
    margin=dict(l=60, r=140, t=90, b=60),
)
fig_coverage.show()
coverage_fraction = len(difference_values) / (len(GRID_X) * len(GRID_Y))
print(f'Common support: {len(difference_values)}/{len(GRID_X) * len(GRID_Y)} grid cells ({100 * coverage_fraction:.1f}%)')
print(f'Median absolute blank/mint neighborhood-count difference on common support: {statistics.median(coverage_differences):.1f} readings')

Common support: 712/780 grid cells (91.3%)
Median absolute blank/mint neighborhood-count difference on common support: 2.0 readings


### 3. Scan order and temporal response

The first two panels color the raw trajectory by progress through each accepted scan window. The third panel shows RMS response over the same normalized progress. Spearman correlations are descriptive diagnostics: repeated readings are not independent observations.

In [9]:
def average_ranks(values):
    order = sorted(range(len(values)), key=values.__getitem__)
    ranks = [0.0] * len(values)
    start = 0
    while start < len(order):
        end = start + 1
        while end < len(order) and values[order[end]] == values[order[start]]:
            end += 1
        average = (start + 1 + end) / 2
        for position in range(start, end):
            ranks[order[position]] = average
        start = end
    return ranks

def pearson(values_a, values_b):
    mean_a, mean_b = statistics.mean(values_a), statistics.mean(values_b)
    centered_a = [value - mean_a for value in values_a]
    centered_b = [value - mean_b for value in values_b]
    denominator = math.sqrt(sum(value * value for value in centered_a) * sum(value * value for value in centered_b))
    return sum(a * b for a, b in zip(centered_a, centered_b)) / denominator if denominator else float('nan')

def spearman(values_a, values_b):
    return pearson(average_ranks(values_a), average_ranks(values_b))

temporal_stats = {}
fig_time = make_subplots(rows=1, cols=3, subplot_titles=('Blank acquisition order', 'Mint acquisition order', 'Response through scan'))
for column, name in enumerate(('Blank', 'Mint'), start=1):
    rows = diagnostic_rows[name]
    fig_time.add_trace(go.Scatter(
        x=[row['x'] for row in rows], y=[row['y'] for row in rows], mode='lines+markers',
        line=dict(color='#A4ACB8', width=1),
        marker=dict(size=7, color=[100 * row['scan_fraction'] for row in rows], coloraxis='coloraxis'),
        customdata=[[row['scan_elapsed_s']] for row in rows],
        hovertemplate='X %{x:.1f} cm<br>Y %{y:.1f} cm<br>Progress %{marker.color:.0f}%<br>Scan time %{customdata[0]:.1f} s<extra></extra>',
        showlegend=False,
    ), row=1, col=column)
    fig_time.update_xaxes(title_text='Desk X (cm)', autorange='reversed', row=1, col=column)
    temporal_stats[name] = {
        'response_time': spearman([row['response'] for row in rows], [row['scan_fraction'] for row in rows]),
        'response_x': spearman([row['response'] for row in rows], [row['x'] for row in rows]),
        'response_y': spearman([row['response'] for row in rows], [row['y'] for row in rows]),
        'time_x': spearman([row['scan_fraction'] for row in rows], [row['x'] for row in rows]),
        'time_y': spearman([row['scan_fraction'] for row in rows], [row['y'] for row in rows]),
    }
for name, style in (('Blank', dict(color='#6B7280', symbol='diamond')), ('Mint', dict(color='#5B6DEE', symbol='circle'))):
    rows = diagnostic_rows[name]
    fig_time.add_trace(go.Scatter(
        x=[100 * row['scan_fraction'] for row in rows], y=[row['response'] for row in rows],
        mode='lines+markers', name=name, line=dict(color=style['color'], width=1.5),
        marker=dict(color=style['color'], symbol=style['symbol'], size=5),
        hovertemplate='Progress %{x:.0f}%<br>RMS %{y:.3f}%<extra>' + name + '</extra>',
    ), row=1, col=3)
fig_time.update_yaxes(title_text='Desk Y (cm)', autorange='reversed', scaleanchor='x', scaleratio=1, row=1, col=1)
fig_time.update_yaxes(autorange='reversed', scaleanchor='x2', scaleratio=1, row=1, col=2)
fig_time.update_xaxes(title_text='Scan progress (%)', row=1, col=3)
fig_time.update_yaxes(title_text='RMS change (%)', row=1, col=3)
fig_time.update_layout(
    title='Acquisition order and sensor persistence', width=1450, height=590, template='plotly_white',
    coloraxis=dict(colorscale='Viridis', cmin=0, cmax=100, colorbar=dict(title='Scan progress (%)', x=0.65)),
    legend=dict(orientation='h', y=1.12, x=0.77), margin=dict(l=65, r=125, t=95, b=60),
)
fig_time.show()
for name, stats in temporal_stats.items():
    print(f"{name}: response-time {stats['response_time']:+.3f} | response-X {stats['response_x']:+.3f} | response-Y {stats['response_y']:+.3f} | time-X {stats['time_x']:+.3f} | time-Y {stats['time_y']:+.3f}")

Blank: response-time +0.825 | response-X -0.814 | response-Y +0.128 | time-X -0.995 | time-Y +0.018
Mint: response-time +0.447 | response-X -0.435 | response-Y -0.253 | time-X -0.981 | time-Y -0.093


### 4. Smoothing sensitivity

The direct subtraction is recalculated at 1.5, 2.0, and 3.0 cm radii. A smaller radius leaves more missing cells but makes fewer assumptions between measurements. Orientation uses the response-weighted principal axis of the strongest positive quarter of each difference map; 0 degrees is horizontal along Desk X and 90 degrees is vertical.

In [10]:
def smooth_response_grid(points, radius_cm):
    sigma_cm = radius_cm * SMOOTH_SIGMA_CM / SMOOTH_RADIUS_CM
    grid = []
    for gy in GRID_Y:
        grid_row = []
        for gx in GRID_X:
            nearby = []
            for point in points:
                distance_sq = (point['x'] - gx) ** 2 + (point['y'] - gy) ** 2
                if distance_sq <= radius_cm ** 2:
                    nearby.append((math.exp(-distance_sq / (2 * sigma_cm ** 2)), point['response']))
            grid_row.append(None if len(nearby) < 2 else sum(weight * value for weight, value in nearby) / sum(weight for weight, _ in nearby))
        grid.append(grid_row)
    return grid

def paired_difference(rows_blank, rows_mint, radius_cm):
    blank = smooth_response_grid(rows_blank, radius_cm)
    mint = smooth_response_grid(rows_mint, radius_cm)
    difference = []
    for blank_row, mint_row in zip(blank, mint):
        difference.append([
            None if blank_value is None or mint_value is None else mint_value - blank_value
            for blank_value, mint_value in zip(blank_row, mint_row)
        ])
    return difference

def hotspot_orientation(grid):
    positive = [value for row in grid for value in row if value is not None and value > 0]
    if len(positive) < 4:
        return {'angle': float('nan'), 'elongation': float('nan'), 'cells': len(positive), 'center_x': float('nan'), 'center_y': float('nan')}
    threshold = percentile(positive, 0.75)
    points = []
    for gy, row in zip(GRID_Y, grid):
        for gx, value in zip(GRID_X, row):
            if value is not None and value >= threshold:
                points.append((gx, gy, max(value - threshold, 1e-9)))
    weight_sum = sum(weight for _, _, weight in points)
    mean_x = sum(x * weight for x, _, weight in points) / weight_sum
    mean_y = sum(y * weight for _, y, weight in points) / weight_sum
    covariance_xx = sum(weight * (x - mean_x) ** 2 for x, _, weight in points) / weight_sum
    covariance_yy = sum(weight * (y - mean_y) ** 2 for _, y, weight in points) / weight_sum
    covariance_xy = sum(weight * (x - mean_x) * (y - mean_y) for x, y, weight in points) / weight_sum
    angle = math.degrees(0.5 * math.atan2(2 * covariance_xy, covariance_xx - covariance_yy))
    if angle > 90:
        angle -= 180
    if angle <= -90:
        angle += 180
    trace = covariance_xx + covariance_yy
    spread = math.sqrt(max(0, ((covariance_xx - covariance_yy) / 2) ** 2 + covariance_xy ** 2))
    major = trace / 2 + spread
    minor = trace / 2 - spread
    elongation = math.sqrt(major / minor) if minor > 0 else float('inf')
    return {'angle': angle, 'elongation': elongation, 'cells': len(points), 'center_x': mean_x, 'center_y': mean_y}

SMOOTHING_RADII = (1.5, 2.0, 3.0)
smoothing_grids = {radius: paired_difference(diagnostic_rows['Blank'], diagnostic_rows['Mint'], radius) for radius in SMOOTHING_RADII}
smoothing_metrics = {}
all_smoothing_values = []
for radius, grid in smoothing_grids.items():
    values = [value for row in grid for value in row if value is not None]
    all_smoothing_values.extend(values)
    smoothing_metrics[radius] = {
        **hotspot_orientation(grid), 'supported': len(values),
        'positive_fraction': sum(value > 0 for value in values) / len(values),
        'median': statistics.median(values),
    }
smoothing_limit = percentile([abs(value) for value in all_smoothing_values], 0.98)
fig_smoothing = make_subplots(rows=1, cols=3, subplot_titles=[
    f'{radius:.1f} cm radius | angle {smoothing_metrics[radius]["angle"]:+.0f} degrees' for radius in SMOOTHING_RADII
])
for column, radius in enumerate(SMOOTHING_RADII, start=1):
    fig_smoothing.add_trace(go.Heatmap(
        x=GRID_X, y=GRID_Y, z=smoothing_grids[radius], coloraxis='coloraxis',
        hovertemplate='X %{x:.1f} cm<br>Y %{y:.1f} cm<br>Mint - blank %{z:+.3f} pp<extra></extra>',
    ), row=1, col=column)
    fig_smoothing.update_xaxes(title_text='Desk X (cm)', autorange='reversed', row=1, col=column)
fig_smoothing.update_yaxes(title_text='Desk Y (cm)', autorange='reversed', scaleanchor='x', scaleratio=1, row=1, col=1)
fig_smoothing.update_yaxes(autorange='reversed', scaleanchor='x2', scaleratio=1, row=1, col=2)
fig_smoothing.update_yaxes(autorange='reversed', scaleanchor='x3', scaleratio=1, row=1, col=3)
fig_smoothing.update_layout(
    title='Mint minus blank across smoothing radii | fixed 3.0 s lag', width=1450, height=600, template='plotly_white',
    coloraxis=dict(colorscale='RdBu_r', cmin=-smoothing_limit, cmax=smoothing_limit, colorbar=dict(title='Difference (pp)')),
    margin=dict(l=60, r=145, t=95, b=60),
)
fig_smoothing.show()
display(Markdown('| Radius | Supported cells | Positive | Median difference | Hotspot center X, Y | Hotspot angle | Elongation |\n|---:|---:|---:|---:|---:|---:|---:|\n' + '\n'.join(
    f"| {radius:.1f} cm | {metrics['supported']} | {100 * metrics['positive_fraction']:.1f}% | {metrics['median']:.3f} pp | {metrics['center_x']:.1f}, {metrics['center_y']:.1f} cm | {metrics['angle']:+.1f} deg | {metrics['elongation']:.2f}x |"
    for radius, metrics in smoothing_metrics.items()
)))

| Radius | Supported cells | Positive | Median difference | Hotspot center X, Y | Hotspot angle | Elongation |
|---:|---:|---:|---:|---:|---:|---:|
| 1.5 cm | 197 | 91.9% | 0.208 pp | 26.1, 31.5 cm | -27.1 deg | 1.30x |
| 2.0 cm | 415 | 95.2% | 0.227 pp | 25.3, 31.9 cm | +4.3 deg | 1.16x |
| 3.0 cm | 712 | 96.3% | 0.215 pp | 24.8, 32.4 cm | +19.5 deg | 1.28x |

### 5. Physical-lag sensitivity

The same 3 cm subtraction is shown at 2.0, 3.0, and 4.0 seconds. The independently estimated 3.0 s lag remains the preregistered working value; the alternatives test whether the geometry is stable rather than selecting the most attractive map.

In [11]:
LAG_VALUES = (2.0, 3.0, 4.0)
lag_grids = {}
lag_metrics = {}
all_lag_values = []
for lag_s in LAG_VALUES:
    rows_blank = corrected_diagnostic_rows(diagnostic_sources['Blank'], lag_s)
    rows_mint = corrected_diagnostic_rows(diagnostic_sources['Mint'], lag_s)
    grid = paired_difference(rows_blank, rows_mint, SMOOTH_RADIUS_CM)
    values = [value for row in grid for value in row if value is not None]
    all_lag_values.extend(values)
    lag_grids[lag_s] = grid
    lag_metrics[lag_s] = {
        **hotspot_orientation(grid), 'supported': len(values),
        'positive_fraction': sum(value > 0 for value in values) / len(values),
        'median': statistics.median(values),
        'blank_rows': len(rows_blank), 'mint_rows': len(rows_mint),
    }
lag_limit = percentile([abs(value) for value in all_lag_values], 0.98)
fig_lag = make_subplots(rows=1, cols=3, subplot_titles=[
    f'{lag_s:.1f} s lag | angle {lag_metrics[lag_s]["angle"]:+.0f} degrees' for lag_s in LAG_VALUES
])
for column, lag_s in enumerate(LAG_VALUES, start=1):
    fig_lag.add_trace(go.Heatmap(
        x=GRID_X, y=GRID_Y, z=lag_grids[lag_s], coloraxis='coloraxis',
        hovertemplate='X %{x:.1f} cm<br>Y %{y:.1f} cm<br>Mint - blank %{z:+.3f} pp<extra></extra>',
    ), row=1, col=column)
    fig_lag.update_xaxes(title_text='Desk X (cm)', autorange='reversed', row=1, col=column)
fig_lag.update_yaxes(title_text='Desk Y (cm)', autorange='reversed', scaleanchor='x', scaleratio=1, row=1, col=1)
fig_lag.update_yaxes(autorange='reversed', scaleanchor='x2', scaleratio=1, row=1, col=2)
fig_lag.update_yaxes(autorange='reversed', scaleanchor='x3', scaleratio=1, row=1, col=3)
fig_lag.update_layout(
    title='Mint minus blank across assumed physical lags | fixed 3 cm smoothing', width=1450, height=600, template='plotly_white',
    coloraxis=dict(colorscale='RdBu_r', cmin=-lag_limit, cmax=lag_limit, colorbar=dict(title='Difference (pp)')),
    margin=dict(l=60, r=145, t=95, b=60),
)
fig_lag.show()
display(Markdown('| Lag | Retained blank / mint | Supported cells | Positive | Median difference | Hotspot center X, Y | Hotspot angle | Elongation |\n|---:|---:|---:|---:|---:|---:|---:|---:|\n' + '\n'.join(
    f"| {lag_s:.1f} s | {metrics['blank_rows']} / {metrics['mint_rows']} | {metrics['supported']} | {100 * metrics['positive_fraction']:.1f}% | {metrics['median']:.3f} pp | {metrics['center_x']:.1f}, {metrics['center_y']:.1f} cm | {metrics['angle']:+.1f} deg | {metrics['elongation']:.2f}x |"
    for lag_s, metrics in lag_metrics.items()
)))

| Lag | Retained blank / mint | Supported cells | Positive | Median difference | Hotspot center X, Y | Hotspot angle | Elongation |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 2.0 s | 174 / 125 | 704 | 95.7% | 0.217 pp | 24.1, 32.2 cm | +3.4 deg | 2.17x |
| 3.0 s | 174 / 126 | 712 | 96.3% | 0.215 pp | 24.8, 32.4 cm | +19.5 deg | 1.28x |
| 4.0 s | 174 / 126 | 712 | 97.3% | 0.227 pp | 25.5, 31.5 cm | +57.4 deg | 1.51x |

### 6. Diagnostic summary

This summary reports stability and temporal confounding without claiming a causal mechanism that this single matched pair cannot identify.

In [12]:
smoothing_angles = [metrics['angle'] for metrics in smoothing_metrics.values()]
lag_angles = [metrics['angle'] for metrics in lag_metrics.values()]
smoothing_span = max(smoothing_angles) - min(smoothing_angles)
lag_span = max(lag_angles) - min(lag_angles)
smoothing_center_span = math.hypot(
    max(metrics['center_x'] for metrics in smoothing_metrics.values()) - min(metrics['center_x'] for metrics in smoothing_metrics.values()),
    max(metrics['center_y'] for metrics in smoothing_metrics.values()) - min(metrics['center_y'] for metrics in smoothing_metrics.values()),
)
lag_center_span = math.hypot(
    max(metrics['center_x'] for metrics in lag_metrics.values()) - min(metrics['center_x'] for metrics in lag_metrics.values()),
    max(metrics['center_y'] for metrics in lag_metrics.values()) - min(metrics['center_y'] for metrics in lag_metrics.values()),
)
mint_time_rho = temporal_stats['Mint']['response_time']
blank_time_rho = temporal_stats['Blank']['response_time']
mint_time_x_rho = temporal_stats['Mint']['time_x']
height_exclusions = {name: [row for row in diagnostic_rows_all_heights[name] if not row['height_in_band']] for name in ('Blank', 'Mint')}
mint_height_excluded = height_exclusions['Mint']
blank_height_excluded = height_exclusions['Blank']

time_statement = (
    'Both blank and mint responses are strongly associated with scan progress, supporting a time-dependent sensor-state, recovery, or drift effect rather than a purely mint-specific spatial feature.'
    if abs(mint_time_rho) >= 0.5 and abs(blank_time_rho) >= 0.5 else
    'Mint response is associated with scan progress, supporting temporal persistence or accumulation as an important contributor.'
    if abs(mint_time_rho) >= 0.3 else
    'Mint response has only a weak monotonic association with scan progress in this run.'
)
smoothing_statement = (
    'The hotspot remains diagonal at every tested smoothing radius, so 3 cm smoothing broadens and rotates it somewhat but does not create it from a horizontal feature.'
    if all(abs(angle) >= 20 for angle in smoothing_angles) else
    'The hotspot orientation crosses near-horizontal under the smoothing sweep, so gridding choices materially affect whether a diagonal is visible.'
)
lag_statement = (
    'A one-second lag change rotates the hotspot substantially; its orientation is therefore not stable enough to claim recovered line direction.'
    if abs(lag_span) >= 20 else
    'The hotspot angle is reasonably stable across the tested lags.'
)
display(Markdown(
    f"**Source loading:** the exact oil dose is unknown and was reported excessive. Mint exceeds background in {100 * positive_fraction:.1f}% of shared cells and the row-profile half-maximum width is {detected_halfmax_width:.1f} cm, so this run tests detectability more cleanly than source-edge sharpness. "
    f"**Height:** {len(blank_height_excluded)} blank and {len(mint_height_excluded)} mint readings were excluded by the 1-3.5 cm band. "
    f"The mint exclusions span Desk Y {min(row['y'] for row in mint_height_excluded):.1f}-{max(row['y'] for row in mint_height_excluded):.1f} cm and are all above 3.5 cm. "
    f"**Coverage:** {100 * coverage_fraction:.1f}% of the predefined grid has common support overall, but smoothing cannot replace the systematically missing high-snout measurements at the source-crossing region. "
    f"**Time:** mint response-versus-progress Spearman rho is {mint_time_rho:+.3f} (blank {blank_time_rho:+.3f}); "
    f"scan progress versus Desk X is {mint_time_x_rho:+.3f}. {time_statement} "
    f"**Smoothing:** hotspot angles span {smoothing_span:.1f} degrees and centers move {smoothing_center_span:.1f} cm. {smoothing_statement} "
    f"**Lag:** hotspot angles span {lag_span:.1f} degrees across 2-4 s while centers move only {lag_center_span:.1f} cm. {lag_statement}"
    + chr(10) * 2
    + "**Most supported diagnosis:** the background and mint-source acquisitions were not height-matched at the source-crossing region: the snout was raised during mint, and the height filter removed those readings. This directly explains the visible mint gaps. The excessive, unknown source dose plausibly broadened the plume, while scan-time confounding and uncertain physical response lag make the remaining hotspot direction unstable. This pair should remain a detectability pilot and should not be used to validate source width or concentration."
))

**Source loading:** the exact oil dose is unknown and was reported excessive. Mint exceeds background in 96.3% of shared cells and the row-profile half-maximum width is 14.0 cm, so this run tests detectability more cleanly than source-edge sharpness. **Height:** 0 blank and 21 mint readings were excluded by the 1-3.5 cm band. The mint exclusions span Desk Y 31.2-43.8 cm and are all above 3.5 cm. **Coverage:** 91.3% of the predefined grid has common support overall, but smoothing cannot replace the systematically missing high-snout measurements at the source-crossing region. **Time:** mint response-versus-progress Spearman rho is +0.447 (blank +0.825); scan progress versus Desk X is -0.981. Mint response is associated with scan progress, supporting temporal persistence or accumulation as an important contributor. **Smoothing:** hotspot angles span 46.5 degrees and centers move 1.6 cm. The hotspot orientation crosses near-horizontal under the smoothing sweep, so gridding choices materially affect whether a diagonal is visible. **Lag:** hotspot angles span 54.0 degrees across 2-4 s while centers move only 1.6 cm. A one-second lag change rotates the hotspot substantially; its orientation is therefore not stable enough to claim recovered line direction.

**Most supported diagnosis:** the background and mint-source acquisitions were not height-matched at the source-crossing region: the snout was raised during mint, and the height filter removed those readings. This directly explains the visible mint gaps. The excessive, unknown source dose plausibly broadened the plume, while scan-time confounding and uncertain physical response lag make the remaining hotspot direction unstable. This pair should remain a detectability pilot and should not be used to validate source width or concentration.